# Appendix: diagnostics and label-free validation

Appendix to the notebook `FOMO.ipynb`. Contents: HMM diagnostics (section 10), label-free validation results (section 11), limitations (section 12), file list (section 13) and the validation code (section 14).

Sections 10–11 only **read** the `validation_report.json` written by `validate.py`; they do not re-run the pipeline. Set `OUT_DIR` below to the folder containing that file.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

OUT_DIR = "outputs/validate"         # folder containing validation_report.json (written by validate.py)
LEAKAGE_CUTOFF = "2023-06-30"     # only used for display if the report does not record the cutoff
P = {"OUT_DIR": OUT_DIR, "LEAKAGE_CUTOFF": LEAKAGE_CUTOFF}

## 10. HMM diagnostics

Diagnostics only, read from `validation_report.json` (written by `validate.py`, part 14). Emission profiles, transition matrix, BIC by K and stability across two seeds.

In [2]:
import json
OUTD = Path(P["OUT_DIR"])
rep = json.load(open(OUTD / "validation_report.json"))
hd = rep["hmm_diagnostics"]
print("K used:", hd["K_used"], "| FOMO level of the states (low -> high):", hd["state_level_sorted"])
print("BIC by K:", hd["bic_by_K"], "| seed stability (ρ):", hd["seed_stability_rho"])
print("ρ(HMM score, feature score):", hd["rho_hmm_vs_feature_score"], "| share of saturated FOMO-state probabilities:", hd["fomo_state_prob_saturated_share"])
print("Transition matrix (row = current state, low -> high):"); display(pd.DataFrame(hd["transition_sorted"]).round(3))
display(pd.DataFrame(hd["state_profile"], index=["low state", "middle state", "high state"][:len(hd["state_level_sorted"])]).T)

K used: 3 | FOMO level of the states (low -> high): [0.203, 0.426, 0.63]
BIC by K: {'2': -350193, '3': -417621, '4': -528679, '5': -572443} | seed stability (ρ): 1.0
ρ(HMM score, feature score): 0.855 | share of saturated FOMO-state probabilities: 0.937
Transition matrix (row = current state, low -> high):


,0,1,2
0,0.481,0.309,0.210
1,0.512,0.303,0.185
2,0.356,0.199,0.445


,low state,middle state,high state
f01_dormancy_before_trigger,0.00,0.38,0.54
f03_price_novelty,0.52,0.64,0.67
f04_trigger_size_uplift,0.00,0.41,0.54
f05_near_peak_dose_response,0.33,0.48,0.52
f06_near_peak_share_excess,0.24,0.47,0.70
f07_near_peak_net_flow,0.13,0.35,0.63
f08_hot_week_burst_ratio,0.10,0.50,0.58
f09_burst_buy_skew,0.00,0.16,0.41
f10_cash_hoard_vs_own,0.00,0.35,0.63
f11_deploy_speed,0.00,0.32,0.64


## 11. Label-free validation results (read from `validation_report.json`)

The full validation suite (`validate.py`, which re-runs the pipeline many times) was run beforehand and wrote `outputs/validate/validation_report.json`. The cells below only read and display those results.

In [3]:
print("11.1 Parameter sensitivity (vs. the base configuration)")
display(pd.DataFrame(rep["parameter_sensitivity"]).T)
print("11.2 Sensitivity to the 6 group weights (200 Dirichlet weight draws, feature-score part only)")
display(pd.Series(rep["weight_sensitivity"]).to_frame("value"))

11.1 Parameter sensitivity (vs. the base configuration)


,investor_rank_corr,level_agreement,top10_overlap,share_high_rows,n_cycles
ONSET_GAP_DAYS=21,0.983,0.898,0.760,0.099,25.0
ONSET_GAP_DAYS=45,0.972,0.881,0.790,0.161,10.0
PEAK_WINDOW=30,0.994,0.937,0.870,0.153,16.0
PEAK_WINDOW=60,0.994,0.949,0.910,0.152,16.0
VOL_WINDOW=60,0.987,0.909,0.800,0.153,15.0
VOL_WINDOW=120,0.997,0.969,0.937,0.156,16.0
DRAWDOWN=0.80,0.994,0.944,0.893,0.157,16.0
HMM_K=2,0.993,0.939,0.897,0.146,16.0
HMM_K=4,0.987,0.924,0.880,0.145,16.0
CORR_TH=0.5,0.993,0.941,0.893,0.148,16.0


11.2 Sensitivity to the 6 group weights (200 Dirichlet weight draws, feature-score part only)


,value
draws,200.000
investor_rank_corr_median,0.996
investor_rank_corr_p5,0.987
top10_overlap_median,0.917


In [4]:
print("11.3 Synthetic investors (generated from behavioural rules, for validation only)")
display(pd.DataFrame(rep["synthetic"]).set_index("type"))
print("11.4 Plausibility")
for k, v in rep["plausibility"].items(): print(" ", k, ":", v)
print("11.5 Edge cases")
display(pd.DataFrame(rep["edge_cases"]).T)
print("11.6 Data leakage (cut at", rep["leakage"].get("cutoff", P["LEAKAGE_CUTOFF"]), "and re-run)")
display(pd.Series(rep["leakage"]).to_frame("result"))

11.3 Synthetic investors (generated from behavioural rules, for validation only)


,mean_score,min_score,max_score,pct_vs_real,share_high
type,,,,,
CONTRA,35.208,27.31,44.54,0.158,0.0
DCA,39.448,35.77,42.10,0.186,0.0
FOMO,58.550,54.32,62.43,0.821,0.1
MOM,55.326,54.18,56.44,0.733,0.0


11.4 Plausibility
  rank_corr_cycle_hot_share_vs_high_share : 0.871
  prox_peak_at_buy_by_level : {'High': 0.958, 'Low': 0.846, 'Medium': 0.919}
  fwd90_return_of_buys_by_level : {'High': 0.1242, 'Low': 0.0854, 'Medium': 0.071}
  panic_sells_per_investor_by_level : {'High': 14.55, 'Low': 2.56, 'Medium': 10.16}
  trait_rank_corr_H1_vs_H2 : 0.83
  lag1_cycle_score_corr : 0.199
  rank_corr_with_previous_design_investor : 0.925
11.5 Edge cases


,rows,mean_score,mean_conf,share_high
no_trades,1647.0,10.15,0.456,0.0000
thin_1_2,9596.0,26.62,0.623,0.0088
sell_only,2593.0,22.58,0.591,0.0000
no_baseline,6074.0,36.18,0.606,0.0530
with_baseline,41332.0,38.07,0.698,0.1550
warmup_cycle,2418.0,40.84,0.715,0.0000
open_cycle,3000.0,42.16,0.770,0.1963
near_threshold,5796.0,45.57,0.467,0.2641


11.6 Data leakage (cut at 2023-06-30 and re-run)


,result
cutoff,2023-06-30
closed cycles compared,10
rows compared,29406
cycle boundaries identical,True
max abs difference,0.0
passed,True


In [5]:
print("11.7 Score scale and thresholds")
h = rep["headline"]
print(" Whole-period score thresholds:", h["inv_thresholds"], "| shares under the dictionary's 40/70 thresholds:", h["dictionary_threshold_shares"])
print("11.8 Logistic regression (explanation only): standardized coefficients")
display(pd.Series(rep["lr_explainer"]["coef"]).sort_values(ascending=False).to_frame("coefficient"))
print(" ρ(LR probability, score) =", rep["lr_explainer"]["rho_lr_prob_vs_score"])
f = rep["features"]
print("11.9 Features: correlation on observed cells vs. with zero-filled cells")
print(" Pairs with |ρ| >= threshold, observed cells:", f["pairs_observed_ge_th"])
print(" Pairs with |ρ| >= threshold, zero-filled cells (comparison only):", len(f["pairs_zero_filled_ge_th"]), "pairs")
display(pd.DataFrame({"NaN share": f["missing_share_nan"], "zero-filled share": f["zero_filled_share"], "eta2": f["eta2_final"],
                      "final weight share": f["weight_share_final"], "ρ with score": f["rho_pct_with_score"]}).round(3))

11.7 Score scale and thresholds
 Whole-period score thresholds: [50.79, 61.22] | shares under the dictionary's 40/70 thresholds: {'rows_gt70': 0.002, 'rows_lt40': 0.52, 'inv_gt70': 0.0, 'inv_lt40': 0.188}
11.8 Logistic regression (explanation only): standardized coefficients


,coefficient
f15_sequence_stages,2.148
f11_deploy_speed,2.067
f01_dormancy_before_trigger,1.438
f09_burst_buy_skew,1.301
f10_cash_hoard_vs_own,1.278
f13_post_peak_silence,1.243
f08_hot_week_burst_ratio,1.226
f03_price_novelty,0.991
f12_hot_notional_excess,0.895
f14_panic_sell_rate,0.830


 ρ(LR probability, score) = 0.952
11.9 Features: correlation on observed cells vs. with zero-filled cells
 Pairs with |ρ| >= threshold, observed cells: [['f02_trigger_vs_own_rate', 'f12_hot_notional_excess', 0.606]]
 Pairs with |ρ| >= threshold, zero-filled cells (comparison only): 16 pairs


,NaN share,zero-filled share,eta2,final weight share,ρ with score
f01_dormancy_before_trigger,0.014,0.475,0.138,0.058,0.817
f02_trigger_vs_own_rate,0.083,0.000,0.131,0.000,0.348
f03_price_novelty,0.162,0.000,0.677,0.022,0.155
f04_trigger_size_uplift,0.003,0.475,0.143,0.058,0.818
f05_near_peak_dose_response,0.481,0.000,0.000,0.067,0.137
f06_near_peak_share_excess,0.183,0.027,0.015,0.066,0.646
f07_near_peak_net_flow,0.240,0.181,0.117,0.059,0.716
f08_hot_week_burst_ratio,0.010,0.378,0.334,0.067,0.685
f09_burst_buy_skew,0.003,0.708,0.067,0.094,0.571
f10_cash_hoard_vs_own,0.059,0.475,0.184,0.055,0.855


## 12. Limitations and what we disclose honestly

- **No ground-truth labels.** Every "agreement" figure (ρ between models, synthetic investors) is a consistency check, not an accuracy measure.
- **Level thresholds changed from 40/70 to percentiles.** On a 0–1 weighted-average scale only 0.2% of rows exceed 70 and 0% of investors exceed 70, so the dictionary's High level is almost empty. We use the 50/90 percentiles (Low = lower half, High = top 10%); the whole-period score thresholds are 50.8 and 61.2. This is a design choice, checked with Q_HIGH = 0.85 / 0.95.
- **f02 is dropped by the correlation step** (|ρ| = 0.606 with f12, just above the 0.6 threshold); results barely change (CORR_TH 0.5 / 0.7 give ρ ≥ 0.993).
- **η² barely changes the score** (without η²: ρ = 0.999); kept because it is a concrete way to implement the requirement to down-weight features driven by the overall market.
- **The HMM adds little beyond the feature score:** ρ(HMM, feature) = 0.855; state probabilities saturate on 94% of rows, so the expected level is used instead of the probability. BIC keeps falling as K grows to 5, so K = 3 is chosen for interpretability (sensitivity at K = 2, 4).
- **f15 (behaviour sequence) has the largest weight share** (18.5%) and is the top signal in 84% of High investor × cycle rows, because it aggregates several steps.
- **The score measures behaviour, not outcomes:** the 90-day return after a buy does not separate the levels clearly.
- **Research grounding:** some thresholds (7 days, 30-day onset gap, 0.97 near-peak, 45 days) are design choices with only indirect support; the parameter table in the report appendix records the source and the level of validation.
- **Simulated data:** no on-chain transactions, so out-of-sample verification is impossible.

## 13. Files in the repository

`README.md` · `validate/fomo_pipeline.py` (all functions, assembled from the cells of sections 1–8) · `validate/params_default.py` · `validate/validate.py` · `validate/synthetic.py` · `data/` (input files) · `outputs/submit/SAMPLE_SUBMISSION.csv` and intermediate tables · `docs/METHODOLOGY.md` (assumptions and method notes).

## 14. Appendix: full validation code

In [6]:
# validate.py: re-runs the pipeline many times (sensitivity, synthetic investors, leakage). Reference only; not run inside the notebook.
VALIDATE_PY = r"""'''Validation of the v2 pipeline (no labels). Writes outputs/validate/validation_report.json.'''
import json, sys, time, warnings, zlib
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
sys.path.insert(0, str(HERE))
from fomo_pipeline import *          # noqa
import matplotlib
matplotlib.use("Agg")
from synthetic import generate       # rule-based archetypes (validation only)
import multiprocessing as mp

ns = {}
exec(open(HERE / "params_default.py").read(), ns)
P = {k: v for k, v in ns.items() if k.isupper()}
OUT = ROOT / P["OUT_DIR"]
for d in ("model", "result", "submit", "validate"): (OUT / d).mkdir(parents=True, exist_ok=True)
market, trades = load_data(ROOT / P["DATA_DIR"])
rep = {}
t0 = time.time()
base = run_pipeline(market, trades, P)
print("base", time.time() - t0, flush=True)
res, inv, w = base["results"], base["investors"], base["windows"]
use = ((res["n_trades"] > 0) & ~res["is_warmup"]).to_numpy()
a = inv.set_index("investor_id")

# 1) parameter sensitivity -------------------------------------------------------------
variants = {"ONSET_GAP_DAYS=21": dict(ONSET_GAP_DAYS=21), "ONSET_GAP_DAYS=45": dict(ONSET_GAP_DAYS=45),
            "PEAK_WINDOW=30": dict(PEAK_WINDOW=30), "PEAK_WINDOW=60": dict(PEAK_WINDOW=60),
            "VOL_WINDOW=60": dict(VOL_WINDOW=60), "VOL_WINDOW=120": dict(VOL_WINDOW=120),
            "DRAWDOWN=0.80": dict(DRAWDOWN=0.80),
            "HMM_K=2": dict(HMM_K=2), "HMM_K=4": dict(HMM_K=4),
            "CORR_TH=0.5": dict(CORR_TH=0.5), "CORR_TH=0.7": dict(CORR_TH=0.7),
            "USE_ETA2=False": dict(USE_ETA2=False),
            "MANUAL_MULT f02,f06,f09=0.5": dict(MANUAL_MULT={"f02_trigger_vs_own_rate": 0.5, "f06_near_peak_share_excess": 0.5, "f09_burst_buy_skew": 0.5}),
            "W_FEATURE=1.0,W_HMM=0.0": dict(W_FEATURE=1.0, W_HMM=0.0),
            "ROLL 0.8/0.2": dict(ROLL_W_INTENSITY=0.8, ROLL_W_RECURRENCE=0.2), "ROLL 0.4/0.6": dict(ROLL_W_INTENSITY=0.4, ROLL_W_RECURRENCE=0.6),
            "TOP_K_CYCLES=2": dict(TOP_K_CYCLES=2),
            "Q_HIGH=0.85": dict(Q_HIGH=0.85), "Q_HIGH=0.95": dict(Q_HIGH=0.95)}
def _variant(item):
    name, ch = item
    o = run_pipeline(market, trades, {**P, **ch})
    b = o["investors"].set_index("investor_id").loc[a.index]
    return name, {"investor_rank_corr": round(float(spearmanr(a["fomo_score"], b["fomo_score"])[0]), 3),
                  "level_agreement": round(float((a["fomo_level"] == b["fomo_level"]).mean()), 3),
                  "top10_overlap": round(float(len(set(a["fomo_score"].nlargest(300).index) & set(b["fomo_score"].nlargest(300).index)) / 300), 3),
                  "share_high_rows": round(float((o["results"].loc[(o["results"]["n_trades"] > 0) & ~o["results"]["is_warmup"], "fomo_level"] == "High").mean()), 3),
                  "n_cycles": int(len(o["cycles"]))}
sens = {}
with mp.get_context("fork").Pool(2) as pool:
    for name, r_ in pool.imap(_variant, list(variants.items())):
        sens[name] = r_; print(name, r_, flush=True)
rep["parameter_sensitivity"] = sens

# 2) group-weight sensitivity (Dirichlet) on the feature part ---------------------------
kmax = max(base["model_log"]); L = base["model_log"][kmax]
act, eta2 = L["active"], pd.Series(L["eta2"])
pctf = res[[f"pct_{f}" for f in FEAT]].copy(); pctf.columns = FEAT
ids = res["investor_id"].to_numpy()
def feat_agg(gw):
    Pw = {**P, **{f"W_{g}": gw[i] for i, g in enumerate(GROUPS)}}
    wt = feature_weights(act, eta2, Pw)
    s_, _, _ = feature_score(pctf, wt)
    return pd.Series(s_[use]).groupby(ids[use]).mean()
bi = feat_agg([P[f"W_{g}"] for g in GROUPS]); k10 = int(0.1 * len(bi)); top0 = set(bi.nlargest(k10).index)
rng = np.random.default_rng(0); rc, ov = [], []
for _ in range(200):
    si = feat_agg(rng.dirichlet(np.full(len(GROUPS), 20 / len(GROUPS))))
    rc.append(spearmanr(bi, si)[0]); ov.append(len(top0 & set(si.nlargest(k10).index)) / k10)
rep["weight_sensitivity"] = {"draws": 200, "investor_rank_corr_median": round(float(np.median(rc)), 3),
                             "investor_rank_corr_p5": round(float(np.percentile(rc, 5)), 3),
                             "top10_overlap_median": round(float(np.median(ov)), 3)}
print(rep["weight_sensitivity"], flush=True)

# 3) HMM diagnostics -----------------------------------------------------------------------
pct = normalize_features(w); inv_code = w["investor_id"].astype("category").cat.codes.to_numpy()
cyc = w["cycle_id"].to_numpy(); N, T = inv_code.max() + 1, int(cyc.max()) + 1
ai = [FEAT.index(f) for f in act]; Xf = np.full((N, T, len(FEAT)), np.nan); Xf[inv_code, cyc] = pct.to_numpy()
Xf = Xf[:, :, ai]; vg = np.zeros((N, T), bool); vg[inv_code, cyc] = True
n_obs = int(vg.sum()); bic = {}
ikey = np.array([zlib.crc32(str(x).encode()) for x in w["investor_id"].astype("category").cat.categories], dtype=np.uint64)
models = {}
for K in (2, 3, 4, 5):
    m = hmm_fit(Xf, vg, K, P["HMM_RESTARTS"], P["RANDOM_STATE"], P["HMM_ITERS"], P["HMM_VAR_FLOOR"], ikey)
    _, ll = hmm_filter(m, Xf, vg); bic[K] = round(hmm_bic(m, ll, n_obs, K, len(act))); models[K] = m
m = models[P["HMM_K"]]; wa = pd.Series(feature_weights(act, eta2, P))[act].to_numpy(); wa = wa / wa.sum()
lev = m["mu"] @ wa; order = np.argsort(lev)
al1, _ = hmm_filter(m, Xf, vg)
m2 = hmm_fit(Xf, vg, P["HMM_K"], P["HMM_RESTARTS"], 7, P["HMM_ITERS"], P["HMM_VAR_FLOOR"], ikey); al2, _ = hmm_filter(m2, Xf, vg)
lev2 = m2["mu"] @ wa
s1 = (al1 @ lev)[vg]; s2 = (al2 @ lev2)[vg]
hs = res.loc[use & res["hmm_fomo_state_prob"].notna().to_numpy(), "hmm_fomo_state_prob"]
rep["hmm_diagnostics"] = {
    "bic_by_K": bic, "K_used": P["HMM_K"], "state_level_sorted": [round(float(lev[i]), 3) for i in order],
    "state_profile": {f: [round(float(m["mu"][i, j]), 2) for i in order] for j, f in enumerate(act)},
    "transition_sorted": [[round(float(m["A"][i, j]), 3) for j in order] for i in order],
    "start_prob_sorted": [round(float(m["pi"][i]), 3) for i in order],
    "seed_stability_rho": round(float(spearmanr(s1, s2)[0]), 3),
    "rho_hmm_vs_feature_score": round(float(spearmanr(res.loc[use, "hmm_score"], res.loc[use, "feature_score"], nan_policy="omit")[0]), 3),
    "fomo_state_prob_saturated_share": round(float(((hs < 0.05) | (hs > 0.95)).mean()), 3)}
print(rep["hmm_diagnostics"], flush=True)

# 4) LR explainer ---------------------------------------------------------------------------
coef, rho_lr = explain_with_lr(res, P)
(OUT / "model" / "lr_coefficients.csv").write_text(coef.sort_values(ascending=False).rename("standardized_coef").to_csv())
rep["lr_explainer"] = {"coef": coef.round(3).to_dict(), "rho_lr_prob_vs_score": round(rho_lr, 3)}

# 5) synthetic archetypes ------------------------------------------------------------------------
syn = generate(market, 20)
o = run_pipeline(market, pd.concat([trades, syn], ignore_index=True), P)
iv = o["investors"]; s = iv[iv["investor_id"].str.startswith("SYN_")].copy()
s["type"] = s["investor_id"].str.split("_").str[1]
real = iv[~iv["investor_id"].str.startswith("SYN_")]["fomo_score"]
s["pct_vs_real"] = s["fomo_score"].apply(lambda v: (real < v).mean())
summ = s.groupby("type").agg(mean_score=("fomo_score", "mean"), min_score=("fomo_score", "min"),
                             max_score=("fomo_score", "max"), pct_vs_real=("pct_vs_real", "mean"),
                             share_high=("fomo_level", lambda x: (x == "High").mean())).round(3)
rep["synthetic"] = summ.reset_index().to_dict("records"); print(summ, flush=True)
s.to_csv(OUT / "validate" / "validation_synthetic_investors.csv", index=False)

# 6) plausibility ---------------------------------------------------------------------------------
cm = base["cycle_mkt"].set_index("cycle_id")
ct = res.groupby("cycle_id").agg(high=("fomo_level", lambda x: (x == "High").mean()))
ct = ct.join(cm)[lambda d: ~d["is_warmup"]]
tl = base["trades"]; close = base["states"]["close"]
fwd = (close.shift(-90) / close - 1)
bb = tl[tl["is_buy"] == 1][["investor_id", "cycle_id", "timestamp", "prox_peak"]].copy()
bb["fwd90"] = fwd.reindex(bb["timestamp"]).to_numpy()
bb = bb.merge(res[["investor_id", "cycle_id", "fomo_level"]], on=["investor_id", "cycle_id"])
ps = tl[tl["panic_sell"] == 1].groupby("investor_id").size()
res["half"] = np.where(res["window_start"] < "2023-07-01", "H1", "H2")
hh = res[use].groupby(["investor_id", "half"])["fomo_score"].mean().unstack()
r3 = res.sort_values(["investor_id", "cycle_id"]); lag = r3.groupby("investor_id")["fomo_score"].shift()
old = Path("../nb2/outputs/investor_level_scores.csv")
cont = {}
if old.exists():
    o_ = pd.read_csv(old).set_index("investor_id")["fomo_score"]
    cont = {"rank_corr_with_previous_design_investor": round(float(spearmanr(a["fomo_score"], o_.loc[a.index])[0]), 3)}
rep["plausibility"] = {
    "rank_corr_cycle_hot_share_vs_high_share": round(float(spearmanr(ct["mkt_hot_share"], ct["high"])[0]), 3),
    "prox_peak_at_buy_by_level": bb.groupby("fomo_level")["prox_peak"].mean().round(3).to_dict(),
    "fwd90_return_of_buys_by_level": bb.groupby("fomo_level")["fwd90"].mean().round(4).to_dict(),
    "panic_sells_per_investor_by_level": inv.assign(p=inv["investor_id"].map(ps).fillna(0)).groupby("fomo_level")["p"].mean().round(2).to_dict(),
    "trait_rank_corr_H1_vs_H2": round(float(spearmanr(hh["H1"], hh["H2"], nan_policy="omit")[0]), 3),
    "lag1_cycle_score_corr": round(float(pd.concat([r3["fomo_score"], lag], axis=1).corr().iloc[0, 1]), 3), **cont}
print(rep["plausibility"], flush=True)

# 7) edge cases ------------------------------------------------------------------------------------
def st(m):
    d = res[m]; return {"rows": int(m.sum()), "mean_score": round(float(d["fomo_score"].mean()), 2),
                        "mean_conf": round(float(d["confidence"].mean()), 3),
                        "share_high": round(float((d["fomo_level"] == "High").mean()), 4)}
rep["edge_cases"] = {"no_trades": st(res["n_trades"] == 0), "thin_1_2": st(res["n_trades"].between(1, 2)),
                     "sell_only": st((res["n_trades"] > 0) & (res["n_buys"] == 0)),
                     "no_baseline": st(~res["has_baseline"]), "with_baseline": st(res["has_baseline"]),
                     "warmup_cycle": st(res["is_warmup"]), "open_cycle": st(res["is_open"]),
                     "near_threshold": st(((res["fomo_score"] - res["thr_low"]).abs().le(1.5)) | ((res["fomo_score"] - res["thr_high"]).abs().le(1.5)))}
print(rep["edge_cases"], flush=True)

# 8) leakage ---------------------------------------------------------------------------------------
rep["leakage"] = {k: (v.item() if hasattr(v, "item") else v) for k, v in
                  leakage_check(market, trades, P, base, P["LEAKAGE_CUTOFF"]).to_dict().items()}
print(rep["leakage"], flush=True)

# 9) feature diagnostics -----------------------------------------------------------------------------
obs = observed_percentiles(w, pct)
Cobs = obs[use].corr(method="spearman", min_periods=P["CORR_MIN_PAIRS"]); Cfill = pct[use].corr(method="spearman")
pairs = lambda C: [[a_, b_, round(float(C.loc[a_, b_]), 3)] for i, a_ in enumerate(FEAT) for b_ in FEAT[i + 1:]
                   if np.isfinite(C.loc[a_, b_]) and abs(C.loc[a_, b_]) >= P["CORR_TH"]]
final_w = pd.Series(L["weights"]); wn = final_w / final_w.sum()
rep["features"] = {
    "active_final": act, "dropped_final": sorted(set(FEAT) - set(act)),
    "pairs_observed_ge_th": pairs(Cobs), "pairs_zero_filled_ge_th": pairs(Cfill),
    "eta2_final": L["eta2"], "weights_final": L["weights"], "weight_share_final": wn.round(3).to_dict(),
    "pruning_by_cycle": {int(k): v["drops"] for k, v in base["model_log"].items()},
    "tag": {f: FEATURES[f][1] for f in FEAT}, "group": {f: FEATURES[f][0] for f in FEAT},
    "missing_share_nan": {f: round(float(pct.loc[use, f].isna().mean()), 3) for f in FEAT},
    "zero_filled_share": {f: round(float(w.loc[use, f"low_{f}"].mean()), 3) for f in FEAT},
    "rho_pct_with_score": {f: round(float(spearmanr(pct.loc[use, f], res.loc[use, "fomo_score"], nan_policy="omit")[0]), 3) for f in FEAT},
    "mean_by_level": res[use].groupby("fomo_level")[[f"pct_{f}" for f in FEAT]].mean().round(3).rename(columns=lambda c: c[4:]).to_dict()}
Cobs.round(3).to_csv(OUT / "model" / "feature_corr_observed.csv"); Cfill.round(3).to_csv(OUT / "model" / "feature_corr_zero_filled.csv")
pd.DataFrame({"feature": FEAT, "group": [FEATURES[f][0] for f in FEAT], "tag": [FEATURES[f][1] for f in FEAT],
              "active": [f in act for f in FEAT], "eta2": [L["eta2"][f] for f in FEAT],
              "weight": [L["weights"][f] for f in FEAT], "weight_share": [round(float(wn[f]), 4) for f in FEAT]}).to_csv(OUT / "model" / "feature_table.csv", index=False)

# key numbers ----------------------------------------------------------------------------------------------
lv = inv["fomo_level"].value_counts().to_dict()
rep["headline"] = {
    "n_cycles": int(len(base["cycles"])), "cycle_len": base["cycles"]["n_days"].describe()[["min", "50%", "max"]].to_dict(),
    "state_share": base["states"].loc[~base["states"]["warmup"], "state"].value_counts(normalize=True).round(3).to_dict(),
    "inv_levels": lv, "inv_thresholds": [round(x, 2) for x in inv.attrs["thresholds"]],
    "cycle_levels_active": res[use]["fomo_level"].value_counts(normalize=True).round(3).to_dict(),
    "dictionary_threshold_shares": {"rows_gt70": round(float((res.loc[use, "fomo_score"] > 70).mean()), 4),
                                    "rows_lt40": round(float((res.loc[use, "fomo_score"] < 40).mean()), 3),
                                    "inv_gt70": round(float((inv["fomo_score"] > 70).mean()), 4),
                                    "inv_lt40": round(float((inv["fomo_score"] < 40).mean()), 3)},
    "score_rows": res.loc[use, "fomo_score"].describe().round(2).to_dict(), "score_investors": inv["fomo_score"].describe().round(2).to_dict(),
    "inv_with_high_cycle": round(float((inv["n_high_cycles"] >= 1).mean()), 3),
    "top1_in_high": res.loc[res["fomo_level"] == "High", "top_1_signal"].value_counts(normalize=True).head(6).round(3).to_dict(),
    "confidence_mean": round(float(res.loc[use, "confidence"].mean()), 3), "confidence_inv_mean": round(float(inv["confidence"].mean()), 3),
    "model_corr_rows": round(float(spearmanr(res.loc[use, "feature_score"], res.loc[use, "hmm_score"], nan_policy="omit")[0]), 3)}
(OUT / "validate" / "validation_report.json").write_text(json.dumps(rep, indent=2, default=str))
res.drop(columns=["half"]).to_csv(OUT / "result" / "investor_cycle_features_scores.csv.gz", index=False, float_format="%.6g")
full, cyc_sub = make_submissions(res, inv)
full.to_csv(OUT / "submit" / "SAMPLE_SUBMISSION.csv", index=False); cyc_sub.to_csv(OUT / "submit" / "submission_cycles.csv", index=False)
inv.to_csv(OUT / "result" / "investor_level_scores.csv", index=False)
base["cycles"].to_csv(OUT / "result" / "cycles.csv", index=False)
import pickle; pickle.dump(dict(inv=inv, res=res, model_log=base["model_log"]), open("final_state.pkl", "wb"))
print("done", time.time() - t0)
"""

In [7]:
# synthetic.py: rule-based investor generator, used for validation only
SYNTHETIC_PY = r"""'''Synthetic archetype investors used ONLY for validation (never for scoring the
real population). They are generated from explicit behavioural rules taken from
the expert narrative, appended to the real trade log, and the full pipeline is
re-run: a sound score must rank FOMO > disciplined momentum > DCA/contrarian.'''
import numpy as np
import pandas as pd

FEE = 0.001


def _ledger(inv, rows, cash0):
    out, cash, pos = [], cash0, 0.0
    for d, side, px, frac in rows:
        if side == "BUY":
            amt = cash * frac / (1 + FEE)
            if amt < 50:
                continue
            q = amt / px; cash -= amt * (1 + FEE); pos += q
        else:
            q = pos * frac
            if q * px < 50:
                continue
            amt = q * px; cash += amt * (1 - FEE); pos -= q
        out.append((inv, d, side, px, q, amt, amt * FEE, cash, pos))
    return out


def generate(market: pd.DataFrame, n_per_type=20, seed=7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    c = market["close"]; days = c.index
    r14 = (c / c.shift(14) - 1).fillna(0).to_numpy()
    prox = (c / c.rolling(90, min_periods=1).max()).to_numpy()
    px = c.to_numpy()
    recs = []
    for k in range(n_per_type):
        # --- FOMO: dormant; when a strong rally lifts price near the peak -> buying burst,
        #     big cash share, consecutive days; later panic-sells in the drawdown
        rows, cool, bought_peak = [], 0, False
        for i, d in enumerate(days):
            if cool > 0:
                cool -= 1; continue
            if r14[i] > 0.15 and prox[i] > 0.96 and rng.random() < 0.5:
                for j in range(rng.integers(3, 6)):
                    if i + j < len(days):
                        rows.append((days[i + j], "BUY", px[i + j], rng.uniform(0.25, 0.45)))
                cool, bought_peak = 45, True
            elif bought_peak and prox[i] < 0.80 and rng.random() < 0.3:
                rows.append((d, "SELL", px[i], rng.uniform(0.4, 0.7))); bought_peak = False; cool = 20
            elif rng.random() < 0.01:
                rows.append((d, "BUY", px[i], rng.uniform(0.03, 0.06)))
        recs += _ledger(f"SYN_FOMO_{k:02d}", rows, rng.uniform(30000, 80000))
        # --- disciplined momentum: same rule in every regime, modest size, exits on downtrends
        rows = []
        for i, d in enumerate(days):
            if i % 5 != k % 5:
                continue
            if r14[i] > 0.05:
                rows.append((d, "BUY", px[i], 0.08))
            elif r14[i] < -0.05:
                rows.append((d, "SELL", px[i], 0.25))
            elif prox[i] > 0.97 and rng.random() < 0.3:
                rows.append((d, "SELL", px[i], 0.15))      # take profit into strength
        recs += _ledger(f"SYN_MOM_{k:02d}", rows, rng.uniform(30000, 80000))
        # --- DCA: fixed-calendar buying, ignores the market
        rows = [(d, "BUY", px[i], 0.04) for i, d in enumerate(days) if i % 14 == k % 14]
        recs += _ledger(f"SYN_DCA_{k:02d}", rows, rng.uniform(30000, 80000))
        # --- contrarian: buys dips, sells rallies
        rows = []
        for i, d in enumerate(days):
            if prox[i] < 0.8 and rng.random() < 0.08:
                rows.append((d, "BUY", px[i], 0.12))
            elif r14[i] > 0.15 and rng.random() < 0.08:
                rows.append((d, "SELL", px[i], 0.2))
        recs += _ledger(f"SYN_CONTRA_{k:02d}", rows, rng.uniform(30000, 80000))
    df = pd.DataFrame(recs, columns=["investor_id", "timestamp", "side", "price", "quantity", "amount",
                                     "fee", "cash_after", "btc_position_after"])
    df.insert(0, "order_id", [f"SYN_{i:08d}" for i in range(len(df))])
    return df
"""